# ResNet9 CNN Examples

In [6]:
from pathlib import Path
import sys

ROOT = Path.cwd()
if not (ROOT / "python").exists() and (ROOT.parent / "python").exists():
    ROOT = ROOT.parent
EXAMPLE_DIR = ROOT / "example_resnet"

PYTHON_DIR = ROOT / "python"
BUILD_DIR = ROOT / "build"
if str(PYTHON_DIR) not in sys.path:
    sys.path.insert(0, str(PYTHON_DIR))
if BUILD_DIR.exists() and str(BUILD_DIR) not in sys.path:
    sys.path.insert(0, str(BUILD_DIR))
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import backend_ndarray as nd  # noqa: E402
import nn  # noqa: E402
from optim import AdamW  # noqa: E402
from example_resnet.resnet import ResNet9, epoch, smoke_train_step, cifar10_loader  # noqa: E402

Pick a backend. If the Metal extension is built and importable, `default_device()` will choose it; otherwise it falls back to NumPy.

In [7]:
device = nd.default_device()
device

metal()

First run a synthetic single-step smoke test. This needs no dataset and is the fastest check that the model can forward, backward, and update parameters.

In [8]:
smoke_train_step(device=device, batch_size=2, seed=0)

{'device': 'metal',
 'logits_shape': (2, 10),
 'loss': 1.3728752136230469,
 'accuracy': 0.5,
 'parameters': 36}

CIFAR-10

In [9]:
from example_resnet import ensure_cifar10

data_dir = ensure_cifar10(EXAMPLE_DIR)

batch_size = 128
epochs = 5
augment = True
data_dir

PosixPath('/Users/mark/Desktop/github/metallic/example_resnet/cifar-10-batches-py')

In [10]:
train_loader = cifar10_loader(
    data_dir,
    train=True,
    batch_size=batch_size,
    device=device,
    augment=augment,
)
x, y = next(iter(train_loader))
print(x.shape, y.shape)

model = ResNet9(device=device)
opt = AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)
history = []
for epoch_idx in range(epochs):
    train_acc, train_loss = epoch(
        train_loader, model, loss_fn=nn.SoftmaxLoss(), opt=opt
    )
    row = {
        "epoch": epoch_idx + 1,
        "train_accuracy": train_acc,
        "train_loss": train_loss,
    }
    history.append(row)
    print(row)
history

(128, 3, 32, 32) (128,)
{'epoch': 1, 'train_accuracy': 0.3226, 'train_loss': 1.8554796978759767}
{'epoch': 2, 'train_accuracy': 0.4211, 'train_loss': 1.5860251706695556}
{'epoch': 3, 'train_accuracy': 0.45872, 'train_loss': 1.4829593313217162}
{'epoch': 4, 'train_accuracy': 0.48134, 'train_loss': 1.4212099979782105}
{'epoch': 5, 'train_accuracy': 0.49906, 'train_loss': 1.3783885333633423}


[{'epoch': 1, 'train_accuracy': 0.3226, 'train_loss': 1.8554796978759767},
 {'epoch': 2, 'train_accuracy': 0.4211, 'train_loss': 1.5860251706695556},
 {'epoch': 3, 'train_accuracy': 0.45872, 'train_loss': 1.4829593313217162},
 {'epoch': 4, 'train_accuracy': 0.48134, 'train_loss': 1.4212099979782105},
 {'epoch': 5, 'train_accuracy': 0.49906, 'train_loss': 1.3783885333633423}]